## Scraping DZ Exams Website to build High School Dataset

In [14]:
import requests
import json
from bs4 import BeautifulSoup as soup
import os
import re
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from webdriver_manager.firefox import GeckoDriverManager
import time
from typing import Optional, Dict
import random


In [15]:
DATA_DIR = "../data/dzexams/maths/"
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

In [16]:

def get_page_html_firefox(url):
    options = Options()
    options.add_argument("--headless")  # run without GUI
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    # Automatically install the right GeckoDriver
    driver = webdriver.Firefox( options=options)
    driver.set_page_load_timeout(20)

    try:
        driver.get(url)
        return driver.page_source
    except Exception as e:
        print("Error:", e)
        return None
    finally:
        driver.quit()
def get_page_html(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # Check for HTTP errors
        return response.text
    except requests.RequestException as e:
        print("Error fetching page:", e)
        return None


In [17]:
# getting main html page for all years 
dz_exams_url = "http://www.dzexams.com/"
base_url = f"{dz_exams_url}ar/"
years_path = [f"{i}as/mathematiques/" for i in range(1, 4)]
years_urls = [f"{base_url}{path}" for path in years_path]
first_year_branches = ["tcst","tcl"]
branches = ["as","al","ge"]
documents = ['rattrapage',"homeworks","cours","exercices","olympiades"]
# all urls 
directories = branches + documents
branches_subdirectories =[f"{branche}_d{i}" for branche in first_year_branches for i in range(1, 4)]+[f"{branche}_e{i}" for branche in first_year_branches for i in range(1, 4)]  + [f"{branche}_d{i}" for branche in branches for i in range(1, 4)] + [f"{branche}_e{i}" for branche in branches for i in range(1, 4)]
all_suffixes = branches_subdirectories + documents
all_urls = [f"{base_url}{year_path}{suffix}" for year_path in years_path for suffix in all_suffixes]
# all_pages = [soup(get_page_html_firefox(url), "html.parser") for url in all_urls]
# page1 = all_pages[0]
page1 = soup(get_page_html_firefox(all_urls[0]))

Error: Message: Navigation timed out after 20000 ms
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:202:5
TimeoutError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:847:5
bail@chrome://remote/content/shared/Sync.sys.mjs:440:19



KeyboardInterrupt: 

In [ ]:
all_urls

['http://www.dzexams.com/ar/1as/mathematiques/tcst_d1',
 'http://www.dzexams.com/ar/1as/mathematiques/tcst_d2',
 'http://www.dzexams.com/ar/1as/mathematiques/tcst_d3',
 'http://www.dzexams.com/ar/1as/mathematiques/tcl_d1',
 'http://www.dzexams.com/ar/1as/mathematiques/tcl_d2',
 'http://www.dzexams.com/ar/1as/mathematiques/tcl_d3',
 'http://www.dzexams.com/ar/1as/mathematiques/tcst_e1',
 'http://www.dzexams.com/ar/1as/mathematiques/tcst_e2',
 'http://www.dzexams.com/ar/1as/mathematiques/tcst_e3',
 'http://www.dzexams.com/ar/1as/mathematiques/tcl_e1',
 'http://www.dzexams.com/ar/1as/mathematiques/tcl_e2',
 'http://www.dzexams.com/ar/1as/mathematiques/tcl_e3',
 'http://www.dzexams.com/ar/1as/mathematiques/as_d1',
 'http://www.dzexams.com/ar/1as/mathematiques/as_d2',
 'http://www.dzexams.com/ar/1as/mathematiques/as_d3',
 'http://www.dzexams.com/ar/1as/mathematiques/al_d1',
 'http://www.dzexams.com/ar/1as/mathematiques/al_d2',
 'http://www.dzexams.com/ar/1as/mathematiques/al_d3',
 'http://w

In [ ]:
page1_links = page1.find_all("a", href=True, class_="btn-item-sujet")
page1_links_urls = [f"{dz_exams_url}{link['href'][1:]}" for link in page1_links]
page1_links_urls

['http://www.dzexams.com/ar/sujets/ajRIT2xLam04YzNDTU4wY3JxRFNPZz09',
 'http://www.dzexams.com/ar/sujets/Z3VqeXJ4Z3hWdTkzM0dMb09QMjJRZz09',
 'http://www.dzexams.com/ar/sujets/dkI2TFRWbFJQQ1hSWlNkYnV1b2Y4UT09',
 'http://www.dzexams.com/ar/sujets/WDlhSUhQS2EzMktoSTZUQnJObzJiUT09',
 'http://www.dzexams.com/ar/sujets/dDF3YmxCUi80QXBOL3EvaHpWQmg1UT09',
 'http://www.dzexams.com/ar/sujets/bDc3ZXFzTGkxNzd6cC9NL0l2bENIQT09',
 'http://www.dzexams.com/ar/sujets/T0NKNkxFcjZ1Q0NESmFSdWdFcnFWZz09',
 'http://www.dzexams.com/ar/sujets/U0QrR3ZDeVNDMll1cjNMaXE0dm9GUT09',
 'http://www.dzexams.com/ar/sujets/bGovMHpkTXQvUjVndjlET0VZeWFKQT09',
 'http://www.dzexams.com/ar/sujets/TUNsZG9jMExOZ0wveDErSUw1WE1FQT09',
 'http://www.dzexams.com/ar/sujets/bzJJZEVJaGh1dGZkWUQyNEtoWEdhZz09',
 'http://www.dzexams.com/ar/sujets/Vm5mc0s2K2JsREk4ODRNbVFJalFWUT09',
 'http://www.dzexams.com/ar/sujets/NEVvazlqZUxsaitHSUFCcUJwaDZ6dz09',
 'http://www.dzexams.com/ar/sujets/dEFlak41L3ZyWm1TVi9jNlFFUlplZz09',
 'http://www.dzexams

In [ ]:
file1 = soup(get_page_html_firefox(page1_links_urls[0]), "html.parser")
file1.find("iframe")["data-src"]

'https://www.dzexams.com/uploads/sujets/sujets2026/dzexams-1as-mathematiques-141137.pdf'

In [ ]:
def download_file(url, save_dir):
    save_path = os.path.join(save_dir, url.split("/")[-1])

    # check if file already exists
    if os.path.exists(save_path):
        print(f"File already exists: {save_path}")
        return
    try:
        response = requests.get(url, stream=True, timeout=10)
        response.raise_for_status()  # Check for HTTP errors
        with open(save_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded: {save_path}")
    except requests.RequestException as e:
        print(f"Error downloading {url}: {e}")
def extract_file_url(page_url):
    page_html = get_page_html_firefox(page_url)
    if not page_html:
        print(f"Failed to retrieve page: {page_url}")
        return None
    page_soup = soup(page_html, "html.parser")
    link = page_soup.find("iframe")['data-src']
    if link:
        return link
    else:
        print(f"No file link found on page: {page_url}")
        return None
    
def extract_and_download_files(page_url, save_dir):
    page_html = get_page_html_firefox(page_url)
    if not page_html:
        print(f"Failed to retrieve page: {page_url}")
        return
    page_soup = soup(page_html, "html.parser")
    links = page_soup.find_all("a", href=True, class_="btn-item-sujet")
    for link in links:
        file_url = extract_file_url(f"{dz_exams_url}{link['href'][1:]}")
        if not file_url:
            continue
        print(f"Extracted file URL: {file_url}")
        download_file(file_url, save_dir)



In [ ]:
for page in all_urls:
    save_path = DATA_DIR
    if  "_" in page.split("/")[-1]:
        branche = page.split("/")[-1].split("_")[0]
        type_ = page.split("/")[-1].split("_")[1][0]
        trimestre = page.split("/")[-1].split("_")[1][-1]
        save_path = os.path.join(save_path, branche, type_, trimestre)
    else:
        save_path = os.path.join(save_path, page.split("/")[-1])
    os.makedirs(save_path, exist_ok=True)
    extract_and_download_files(page, save_path)
    

Error: Message: Reached error page: about:neterror?e=dnsNotFound&u=http%3A//www.dzexams.com/ar/1as/mathematiques/tcst_d1&c=UTF-8&d=We%20can%E2%80%99t%20connect%20to%20the%20server%20at%20www.dzexams.com.&a=
Stacktrace:
RemoteError@chrome://remote/content/shared/RemoteError.sys.mjs:8:8
WebDriverError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:202:5
UnknownError@chrome://remote/content/shared/webdriver/Errors.sys.mjs:961:5
checkReadyState@chrome://remote/content/marionette/navigate.sys.mjs:59:24
onNavigation@chrome://remote/content/marionette/navigate.sys.mjs:351:39
emit@resource://gre/modules/EventEmitter.sys.mjs:156:19
receiveMessage@chrome://remote/content/marionette/actors/MarionetteEventsParent.sys.mjs:33:25

Failed to retrieve page: http://www.dzexams.com/ar/1as/mathematiques/tcst_d1


KeyboardInterrupt: 